# Lab Day 2 — Track 4 · Nguyễn Phúc Huy · 2A202602911

Chạy trên **Kaggle**. Xem `README.md` trong thư mục nộp để biết cách chạy lại.
Cell dưới kiểm tra Internet/GPU rồi **khôi phục tiến độ từ output của kernel trước**
(notebook cache-aware nên mọi thí nghiệm đã xong sẽ được bỏ qua).


In [ ]:
import glob, os, platform, shutil, socket, subprocess, sys
from pathlib import Path
os.environ["LAB_STAGE"] = "all"
os.environ["LAB_SEEDS"] = "0,1,2"
os.environ["LAB_BUDGET_MIN"] = "99999"      # phiên Kaggle dài -> tắt "dừng mềm"

REPO_URL = "https://github.com/Yuhnguyn/K4-Track4-NguyenPhucHuy-2A202602911-Day2-Deeplearning-Advance.git"
SUB_NAME = "2A202602911_nguyenphuchuy"
REPO_DIR = Path("/kaggle/working/lab")
SUB = REPO_DIR / "submissions" / SUB_NAME

print("cwd:", os.getcwd(), "| python", platform.python_version())

def _dns(host):
    try:
        socket.gethostbyname(host)
        return True
    except Exception:
        return False

_net = _dns("pypi.org") and _dns("github.com") and _dns("zenodo.org")
_r = subprocess.run([sys.executable, "-c",
                     "import torch;print('torch',torch.__version__,'cuda',torch.cuda.is_available(),"
                     "torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO-GPU')"],
                    capture_output=True, text=True)
print("Internet:", "BẬT" if _net else "TẮT", "|", _r.stdout.strip() or _r.stderr.strip()[-200:])
if not _net:
    raise SystemExit("\n*** DỪNG: notebook Kaggle KHÔNG có Internet ***\nXác minh số điện thoại tại "
                     "https://www.kaggle.com/settings rồi bật Internet trong Settings của notebook.")
if "cuda True" not in _r.stdout:
    print("\n*** CẢNH BÁO: chưa thấy CUDA — kiểm tra Settings -> Accelerator = GPU ***\n")

# --- clone repo SỚM (để cell 2 thấy repo đã có -> bỏ qua clone) rồi chép đè artifact khôi phục ---
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    print("đã clone repo ->", REPO_DIR)

# Nguồn khôi phục: output của kernel cũ được mount ở /kaggle/input/<slug>/
cands = sorted(Path("/kaggle/input").glob("*/lab/submissions/" + SUB_NAME))
cands += sorted(Path("/kaggle/input").glob("*/submissions/" + SUB_NAME))
print("nguồn khôi phục:", [str(c) for c in cands] or "(không có — chạy từ đầu)")
if cands:
    src = cands[0]
    for d in ("runs", "curves", "predictions", "eval_out"):
        if (src / d).exists():
            shutil.copytree(src / d, SUB / d, dirs_exist_ok=True)
            print(f"  + {d}/:", sum(1 for _ in (src / d).rglob('*') if _.is_file()), "file")
    for f in ("runs_summary.csv", "inference_val.csv", "inference_choice.json", "latency.csv",
              "latency_results.json", "results.xlsx", "report.md"):
        if (src / f).exists():
            shutil.copy2(src / f, SUB / f)
            print("  +", f)
    done = sorted(p.parent.parent.name + "/seed" + p.parent.name[4:]
                  for p in (SUB / "runs").glob("*/seed*/done.json"))
    print(f"  => {len(done)} thí nghiệm đã xong, notebook sẽ bỏ qua:")
    print("     " + ", ".join(done))


# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

**Nguyễn Phúc Huy · 2A202602911 · Track 4 · Ngày 2**

Notebook này đã hoàn thiện bộ khung `starter/` (thành `code/`) và chạy đủ 5 bước:
Bước 0 (EDA + kiểm tra pipeline) → Bước 1 (≥ 5 backbone) → Bước 2 (≥ 3 trục công thức huấn luyện)
→ Bước 3 (≥ 4 phương pháp suy luận + đo độ trễ) → Bước 4 (chung kết ≥ 3 seed, test một lần mỗi seed)
→ Bước 5 (`results.xlsx`, `curves/`, `report.md`).

**Hai quy tắc không được vi phạm**

1. Mọi lựa chọn (backbone, siêu tham số, phương pháp suy luận, nhiệt độ T) chỉ dựa trên **val**.
2. **Test chỉ mở ở Bước 4**, đúng một lần cho mỗi seed (`save_test_predictions=True`).

**Cách chạy lại:** *Runtime → Change runtime type → T4 GPU* rồi *Run all*. Mọi bước đều
**cache-aware** (đọc `runs/<exp_id>/seed<k>/done.json`): chạy lại chỉ huấn luyện phần còn thiếu.
Biến môi trường `LAB_STAGE` (`b0`/`b1`/`b2`/`b3`/`b4`/`b5`, mặc định `all`) cho phép chạy từng bước.


## 0. Cài đặt môi trường


In [ ]:
!pip -q install timm openpyxl fvcore

import hashlib, json, os, platform, shutil, subprocess, sys, tarfile, time, warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)

REPO_URL = "https://github.com/Yuhnguyn/K4-Track4-NguyenPhucHuy-2A202602911-Day2-Deeplearning-Advance.git"
SUB_NAME = "2A202602911_nguyenphuchuy"
STAGE = os.environ.get("LAB_STAGE", "all")
# Kaggle có cả /content (không ghi được) -> phải ưu tiên /kaggle/working
ON_KAGGLE = Path("/kaggle/working").is_dir()
ON_COLAB = (not ON_KAGGLE) and Path("/content").is_dir()
REPO_DIR = (Path("/kaggle/working/lab") if ON_KAGGLE else
            (Path("/content/lab") if ON_COLAB else Path.cwd().parent))

def stage(*names):
    return STAGE in ("all", *names)


# LAB_ONLY="B01,B02" -> chỉ chạy đúng các exp_id này (để chia nhỏ job theo từng lần chạy khi
# phiên Colab ngắn); rỗng = chạy tất cả.
ONLY = [s.strip() for s in os.environ.get("LAB_ONLY", "").split(",") if s.strip()]


def want(exp_id):
    return (not ONLY) or (exp_id in ONLY)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
print("repo:", REPO_DIR, "| stage:", STAGE, "| python", platform.python_version())

SUB = REPO_DIR / "submissions" / SUB_NAME
for d in ("code", "curves", "predictions", "data/labels", "runs"):
    (SUB / d).mkdir(parents=True, exist_ok=True)

# Chép starter/ -> code/ CHỈ KHI chưa có (không ghi đè bản đã hoàn thiện).
for f in sorted((REPO_DIR / "starter").glob("*")):
    dst = SUB / "code" / f.name
    if not dst.exists():
        shutil.copy2(f, dst)
# Bản sao NGUYÊN VẸN của eval.py (không sửa) để thư mục nộp tự chứa; nếu repo chưa có thì bỏ qua.
src_eval, dst_eval = REPO_DIR / "eval.py", SUB / "code" / "eval.py"
if src_eval.exists() and (not dst_eval.exists() or
                          hashlib.sha256(src_eval.read_bytes()).hexdigest() !=
                          hashlib.sha256(dst_eval.read_bytes()).hexdigest()):
    if not dst_eval.exists():
        shutil.copy2(src_eval, dst_eval)

os.chdir(SUB)
sys.path.insert(0, str(SUB / "code"))
sys.path.insert(0, str(REPO_DIR))
print("thư mục làm việc:", Path.cwd())
print("code/:", sorted(p.name for p in (SUB / "code").glob("*")))


### Tải dữ liệu (ảnh Zenodo + nhãn/chia fold của tác giả)

Ảnh ~490 MB, kiểm tra MD5. Giải nén ra **đĩa cục bộ** của VM (17.509 ảnh nhỏ: đọc từ Drive sẽ rất chậm).


In [ ]:
DATA = SUB / "data"
LABELS_DIR = DATA / "labels"
LABELS_DIR.mkdir(parents=True, exist_ok=True)
MD5 = "b7b30f96d466fba86016aa5a26606e0f"
zip_path = DATA / "images.zip"

if not zip_path.exists():
    subprocess.run(["wget", "-q", "-O", str(zip_path),
                    "https://zenodo.org/records/7939060/files/images.zip?download=1"], check=True)
h = hashlib.md5()
with open(zip_path, "rb") as fh:
    for chunk in iter(lambda: fh.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == MD5, f"MD5 sai: {h.hexdigest()} (kỳ vọng {MD5})"
print(f"images.zip OK: {zip_path.stat().st_size / 1e6:.1f} MB, MD5 {h.hexdigest()}")

if not any(DATA.rglob("*.jpg")):
    subprocess.run(["unzip", "-q", "-n", str(zip_path), "-d", str(DATA)], check=True)

# Tự dò thư mục chứa ảnh (zip có thể giải nén ra data/images hoặc thẳng data/)
IMAGES_DIR = DATA / "images"
if not any(IMAGES_DIR.glob("*.jpg")):
    sub = DATA / "images"
    lotus = [p.parent for p in DATA.glob("*/images/*.jpg")][:1]
    IMAGES_DIR = lotus[0] if lotus else DATA
n_img = sum(1 for _ in IMAGES_DIR.glob("*.jpg"))
print("IMAGES_DIR =", IMAGES_DIR, "|", n_img, "ảnh .jpg")
assert n_img == 17509, f"số ảnh .jpg = {n_img}, kỳ vọng 17509"

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ("labels", "train_subset0", "val_subset0", "test_subset0"):
    dst = LABELS_DIR / f"{name}.csv"
    if not dst.exists():
        subprocess.run(["wget", "-q", "-O", str(dst), f"{BASE}/{name}.csv"], check=True)
print("labels:", sorted(p.name for p in LABELS_DIR.glob("*.csv")))


### Helper: cache-aware, registry, backup — chạy theo mảnh ngắn để sống sót khi Colab thu hồi VM

Colab free **không công bố** thời gian sống tối đa của VM (FAQ: *usage limits as well as idle
timeout periods, maximum VM lifetime … vary over time*, free = tối đa 12 giờ nhưng thường bị thu hồi
sớm hơn nhiều), nên không thể trông vào một phiên dài. Notebook vì vậy chạy theo **mảnh ngắn**:

| Biến môi trường | Ý nghĩa | Mặc định |
|---|---|---|
| `LAB_BUDGET_MIN` | Dừng **mềm** sau ~N phút huấn luyện trong một lần chạy cell | `22` |
| `LAB_STAGE` | Chỉ chạy một bước: `b0`…`b5` | `all` |
| `LAB_ONLY` | Chỉ chạy đúng các `exp_id` này (vd `B01,B02`) | rỗng = tất cả |
| `LAB_SEEDS` | Seed cho Bước 4 | `0,1,2` |
| `LAB_SYNC_PT` | Đưa cả `runs/B*/seed*/best.pt` lên Drive (Bước 3 cần nạp lại) | `1` |
| `LAB_BACKUP_DIR` | Đổi thư mục backup | `<MyDrive>/lab_day2_ckpt` |

**Cách chạy khi phiên ngắn:** cứ bấm *Run all* (hoặc chạy cell của bước đang làm). Sau **mỗi** thí
nghiệm notebook tự chép kết quả (kể cả checkpoint) ra Drive; khi gần hết ngân sách nó **tự dừng
sạch** thay vì để Colab giết giữa chừng. Phiên sau mở lại: mọi thí nghiệm đã xong được bỏ qua tự
động (cache `runs/<exp_id>/seed<k>/done.json`). Không phải nhớ mình đang ở đâu.


In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import dataset, losses, model as model_mod, train, inference, benchmark
import eval as ev
from train import Config, run

CURVES, PREDS, RUNS = SUB / "curves", SUB / "predictions", SUB / "runs"
REGISTRY = SUB / "runs_summary.csv"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import timm
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")
print("torch", torch.__version__, "| timm", timm.__version__)


# --------------------------------------------------------------------------- #
# Sống sót qua việc Colab thu hồi VM: lưu tiến độ ra Google Drive (repo README mục 4
# yêu cầu: "lưu mọi log, đường cong, logit dự đoán và checkpoint ra ổ bền NGAY SAU MỖI LẦN CHẠY").
# --------------------------------------------------------------------------- #
BUDGET_MIN = float(os.environ.get("LAB_BUDGET_MIN", "22"))   # dừng mềm sau ~22 phút huấn luyện
SYNC_PT = os.environ.get("LAB_SYNC_PT", "1") != "0"
_T_FIRST = None        # mốc huấn luyện đầu tiên trong phiên (tính ngân sách)
_LAST_MIN = None       # thời gian (phút) của lần chạy gần nhất -> ước lượng cho lần sau
_budget_hit = False


def _backup_root():
    env = os.environ.get("LAB_BACKUP_DIR")
    if env:
        return Path(env)
    if Path("/content/drive/MyDrive").is_dir():
        return Path("/content/drive/MyDrive/lab_day2_ckpt")
    return None


BACKUP_DIR = _backup_root()


def drive_mount():
    """Mount Google Drive (chỉ có tác dụng trong Colab giao diện). Không mount được thì bỏ qua."""
    global BACKUP_DIR
    try:
        from google.colab import drive  # noqa: F401
    except Exception:
        print("[drive] không phải Colab (hoặc thiếu google.colab) — bỏ qua mount Drive")
        BACKUP_DIR = _backup_root()
        return BACKUP_DIR
    try:
        if not Path("/content/drive/MyDrive").is_dir():
            drive.mount("/content/drive")
    except Exception as exc:
        print("[drive] mount thất bại:", exc)
    BACKUP_DIR = _backup_root()
    print("[drive] thư mục lưu tiến độ:", BACKUP_DIR)
    return BACKUP_DIR


def _keep(rel: Path) -> bool:
    """File nào đưa lên Drive: metadata/logit/curves/predictions/eval_out + checkpoint nhóm B
    (Bước 3 phải nạp lại best.pt). Bỏ ảnh dữ liệu, __pycache__, file gói .tar.gz/.zip."""
    name = rel.name
    if name.endswith((".pt", ".pth", ".ckpt")):
        return bool(SYNC_PT) and len(rel.parts) > 1 and rel.parts[0] == "runs" \
            and rel.parts[1].startswith("B")
    if "__pycache__" in rel.parts or name.endswith((".pyc", ".tar.gz", ".zip")):
        return False
    if rel.parts and rel.parts[0] == "data":
        return False
    return True


def _copy_if_newer(src: Path, dst: Path) -> int:
    st = src.stat()
    if dst.exists() and dst.stat().st_size == st.st_size and int(dst.stat().st_mtime) >= int(st.st_mtime):
        return 0
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return 1


def sync_out(tag: str = "") -> int:
    """Chép trạng thái khôi phục được sang BACKUP_DIR. Gọi TỰ ĐỘNG sau mỗi lần huấn luyện
    -> VM chết giữa chừng thì mất tối đa một thí nghiệm."""
    if BACKUP_DIR is None:
        return 0
    n = 0
    for top in ("runs", "curves", "predictions", "eval_out"):
        root = SUB / top
        if not root.exists():
            continue
        for p in sorted(root.rglob("*")):
            if p.is_file() and _keep(p.relative_to(SUB)):
                n += _copy_if_newer(p, BACKUP_DIR / p.relative_to(SUB))
    for name in ("runs_summary.csv", "inference_val.csv", "inference_choice.json", "latency.csv",
                 "latency_results.json", "results.xlsx", "report.md", "README.md"):
        p = SUB / name
        if p.exists():
            n += _copy_if_newer(p, BACKUP_DIR / name)
    if n:
        print(f"  [drive] +{n} file -> {BACKUP_DIR}" + (f"  ({tag})" if tag else ""))
    return n


def sync_in(verbose: bool = True) -> int:
    """Khôi phục tiến độ từ BACKUP_DIR (chạy ở đầu mỗi phiên mới, sau khi đã có SUB)."""
    if BACKUP_DIR is None or not BACKUP_DIR.exists():
        if verbose:
            print("[drive] chưa có backup (phiên đầu) — bỏ qua khôi phục")
        return 0
    n = 0
    for p in sorted(BACKUP_DIR.rglob("*")):
        if p.is_file():
            n += _copy_if_newer(p, SUB / p.relative_to(BACKUP_DIR))
    if verbose:
        print(f"[drive] khôi phục {n} file từ {BACKUP_DIR} "
              f"| số lần chạy đã xong: {len(list(RUNS.glob('*/seed*/done.json')))}")
    return n


def run_cached(cfg, force=False):
    """Chạy train.run với 3 tính chất sống-sót-qua-phiên:

      1. cache-aware  — có runs/<exp_id>/seed<k>/done.json thì bỏ qua, trả về kết quả cũ;
      2. lưu ngay     — xong là chép kết quả (kể cả best.pt của nhóm B) ra Drive;
      3. dừng mềm     — nếu thời gian còn lại không đủ cho một thí nghiệm nữa thì trả về None
                        (KHÔNG huấn luyện), để cell kết thúc sạch trước khi Colab thu hồi VM.
    """
    global _T_FIRST, _LAST_MIN, _budget_hit
    d = train.run_dir(cfg) / "done.json"
    if d.exists() and not force:
        s = json.loads(d.read_text("utf-8"))
        print(f"[cache] {cfg.exp_id} seed{cfg.seed} {cfg.backbone}: "
              f"val macro-F1 {s['val_macro_f1']:.4f} (best epoch {s['best_epoch']})")
        return s
    if _T_FIRST is None:
        _T_FIRST = time.time()
    left_min = BUDGET_MIN - (time.time() - _T_FIRST) / 60
    need_min = max(3.0, (_LAST_MIN or 10.0) * 1.15)     # ước lượng 1 thí nghiệm + 15% biên
    if left_min < need_min:
        if not _budget_hit:
            _budget_hit = True
            print(f"\n⏸  DỪNG MỀM: còn {left_min:.1f} phút, cần ~{need_min:.1f} phút cho thí nghiệm kế "
                  f"tiếp (ngân sách LAB_BUDGET_MIN={BUDGET_MIN:.0f}).")
            print(f"   Tiến độ đã lưu ở: {BACKUP_DIR or '(chưa bật backup — kiểm tra drive_mount)'}")
            print("   Phiên sau: mở lại notebook và chạy lại ĐÚNG cell này — thí nghiệm đã xong "
                  "được bỏ qua tự động.")
        return None
    t0 = time.perf_counter()
    s = run(cfg)
    _LAST_MIN = (time.perf_counter() - t0) / 60
    print(f"[xong] {cfg.exp_id} seed{cfg.seed} sau {_LAST_MIN:.1f} phút")
    sync_out(f"{cfg.exp_id}_seed{cfg.seed}")
    return s


def registry() -> pd.DataFrame:
    """Bảng mọi lần chạy đã hoàn tất (đọc từ done.json) - nguồn số liệu cho results.xlsx."""
    rows = []
    for p in sorted(RUNS.glob("*/seed*/done.json")):
        s = json.loads(p.read_text("utf-8"))
        s.pop("history", None)
        s["run_dir"] = str(p.parent.relative_to(SUB))
        rows.append(s)
    df = pd.DataFrame(rows)
    if len(df):
        df.to_csv(REGISTRY, index=False)
    return df


def history_of(exp_id, seed=0) -> pd.DataFrame:
    return pd.read_csv(train.run_dir(Config(exp_id=exp_id, seed=seed)) / "history.csv")


def backup(tag="latest"):
    """Đóng gói artifact (bỏ *.pt nặng) để tải về máy / khôi phục ở session sau."""
    out = SUB / f"artifacts_{tag}.tar.gz"
    with tarfile.open(out, "w:gz") as t:
        for d in ("runs", "curves", "predictions"):
            if (SUB / d).exists():
                t.add(SUB / d, arcname=d,
                      filter=lambda ti: None if ti.name.endswith((".pt", ".pth")) else ti)
        for f in ("runs_summary.csv", "inference_results.json", "latency_results.json",
                  "results.xlsx", "report.md"):
            if (SUB / f).exists():
                t.add(SUB / f, arcname=f)
    print("backup:", out, f"{out.stat().st_size / 1e6:.1f} MB")
    return out


def restore(src):
    """Khôi phục artifact từ file .tar.gz (gọi ở đầu session sau khi colab bị ngắt)."""
    if Path(src).exists():
        with tarfile.open(src) as t:
            t.extractall(SUB)
        print("đã khôi phục từ", src)
    else:
        print("không có file khôi phục:", src)


# --------------------------------------------------------------------------- #
# Trạng thái suy ra TỪ ĐĨA: đủ để chạy riêng một stage (b1/b2/b3/b4/b5) sau khi
# session Colab bị ngắt, không phụ thuộc biến của các cell trước.
# --------------------------------------------------------------------------- #
T00_KW = dict(epochs=10, batch_size=64, lr_backbone=1e-4, lr_head=1e-3, weight_decay=0.05,
              warmup_epochs=1.0, amp=True, img_size=224, aug="basic", init="finetune",
              loss="ce", sampler=None, mix=None, ema_decay=None, num_workers=2,
              images_dir=str(IMAGES_DIR), labels_dir=str(LABELS_DIR))

train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)


def pick_main_backbone(reg):
    """Backbone CNN tốt nhất theo macro-F1 val. Bỏ nhóm transformer vì TTA nhiều crop / nhiều độ
    phân giải và gộp BatchNorm ở Bước 3 không chạy được với lưới patch cố định của ViT/Swin."""
    b = reg[reg["exp_id"].str.startswith("B")].sort_values("val_macro_f1", ascending=False)
    cnn = b[~b["backbone"].str.contains("deit|vit|swin", case=False, regex=True)]
    return (cnn if len(cnn) else b).iloc[0]


drive_mount()
sync_in()

_reg = registry()
if len(_reg):
    MAIN = pick_main_backbone(_reg)
    if (SUB / "inference_choice.json").exists():
        METHOD = json.loads((SUB / "inference_choice.json").read_text("utf-8"))["method"]
    print("khôi phục trạng thái: MAIN =", MAIN["exp_id"], MAIN["backbone"],
          "| METHOD =", globals().get("METHOD", "(chưa chốt)"))


## Bước 0 — EDA và kiểm tra pipeline

Mục tiêu: hiểu dữ liệu, chạy **đủ các kiểm tra bắt buộc ở README.md mục 2.1** (số ảnh mỗi tập/mỗi lớp,
giao rỗng, hợp đủ 17.509 ảnh, mọi file tồn tại), rồi kiểm tra pipeline trước khi chạy thật.


In [ ]:
if not stage("b0"):
    print("bỏ qua Bước 0")
else:
    train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
    split_info = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR, verbose=True)

    # Biểu đồ phân bố lớp (Hình 1 của báo cáo)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
    x = np.arange(dataset.NUM_CLASSES)
    w = 0.27
    for i, (name, df) in enumerate((("train", train_df), ("val", val_df), ("test", test_df))):
        counts = df["Label"].value_counts().reindex(range(dataset.NUM_CLASSES), fill_value=0)
        axes[0].bar(x + (i - 1) * w, counts.to_numpy(), width=w, label=name)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(dataset.CLASS_NAMES, rotation=40, ha="right", fontsize=8)
    axes[0].set_ylabel("số ảnh"); axes[0].set_title("Phân bố lớp — fold 0"); axes[0].legend()
    axes[0].grid(alpha=.3, axis="y")

    all_counts = pd.concat([train_df, val_df, test_df])["Label"].value_counts() \
                   .reindex(range(dataset.NUM_CLASSES), fill_value=0)
    axes[1].bar(x, all_counts.to_numpy(), color="tab:orange")
    for i, v in enumerate(all_counts.to_numpy()):
        axes[1].text(i, v + 60, str(v), ha="center", fontsize=8)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(dataset.CLASS_NAMES, rotation=40, ha="right", fontsize=8)
    axes[1].set_ylabel("số ảnh"); axes[1].set_title("Toàn dataset (17.509 ảnh)")
    axes[1].grid(alpha=.3, axis="y")
    fig.tight_layout(); fig.savefig(CURVES / "EDA_class_distribution.png", dpi=140); plt.close(fig)

    # Ảnh mẫu: 3 ảnh mỗi lớp
    fig, axes = plt.subplots(dataset.NUM_CLASSES, 3, figsize=(5.2, 2.0 * dataset.NUM_CLASSES))
    for c in range(dataset.NUM_CLASSES):
        files = train_df.loc[train_df["Label"] == c, "Filename"].head(3).tolist()
        for j, fn in enumerate(files):
            axes[c, j].imshow(plt.imread(IMAGES_DIR / fn))
            axes[c, j].axis("off")
            if j == 0:
                axes[c, j].set_title(f"{dataset.CLASS_NAMES[c]}", fontsize=8, loc="left")
    fig.tight_layout(); fig.savefig(CURVES / "EDA_samples.png", dpi=120); plt.close(fig)

    imbalance = float(all_counts.max() / all_counts.min())
    print(f"\nTỉ lệ mất cân bằng (lớp lớn nhất / nhỏ nhất) = {imbalance:.2f}")
    print("Đối chiếu Table 1 bài báo: Negative 9.106; các loài 1.009-1.125 ảnh")


In [ ]:
if not stage("b0"):
    print("bỏ qua kiểm tra pipeline")
else:
    # 1) cố định seed
    train.set_seed(0)

    # 2) loss ban đầu của head mới phải xấp xỉ -ln(1/9) = 2.197
    m = model_mod.build_model("resnet18", pretrained=False, num_classes=dataset.NUM_CLASSES)
    x = torch.randn(8, 3, 64, 64)
    y = torch.randint(0, dataset.NUM_CLASSES, (8,))
    crit = torch.nn.CrossEntropyLoss()
    with torch.no_grad():
        step0 = float(crit(m(x), y))
    ln9 = float(np.log(dataset.NUM_CLASSES))
    print(f"[2] loss bước 0 = {step0:.4f} | -ln(1/9) = {ln9:.4f} | lệch {abs(step0 - ln9):.4f}")

    # 3) overfit một batch nhỏ (10 mẫu) tới loss gần 0
    m2 = model_mod.build_model("resnet18", pretrained=False, num_classes=dataset.NUM_CLASSES)
    opt = torch.optim.AdamW(m2.parameters(), lr=1e-3)
    xb = torch.randn(10, 3, 64, 64); yb = torch.randint(0, dataset.NUM_CLASSES, (10,))
    m2.train()
    l0 = None
    for it in range(120):
        loss = crit(m2(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        l0 = float(loss.detach())
    with torch.no_grad():
        acc = float((m2(xb).argmax(1) == yb).float().mean())
    print(f"[3] overfit 10 mẫu: loss {l0:.5f}, accuracy trên chính batch đó {acc:.2f}")

    # 4) ảnh sau augmentation (đã giải chuẩn hoá) + nhãn, để chắc ảnh và nhãn khớp nhau
    tf = dataset.build_transforms(True, img_size=224, aug="basic")
    ds_big = dataset.DeepWeedsDataset(train_df.head(9), IMAGES_DIR, tf)
    mean, std = np.array(dataset.IMAGENET_MEAN), np.array(dataset.IMAGENET_STD)
    fig, axes = plt.subplots(3, 3, figsize=(6.6, 6.8))
    for i in range(9):
        img, lab, fn = ds_big[i]
        arr = (img.permute(1, 2, 0).numpy() * std + mean).clip(0, 1)
        axes[i // 3, i % 3].imshow(arr); axes[i // 3, i % 3].axis("off")
        axes[i // 3, i % 3].set_title(f"{dataset.CLASS_NAMES[lab]}", fontsize=7)
    fig.suptitle("Ảnh train sau augmentation (giải chuẩn hoá)")
    fig.tight_layout(); fig.savefig(CURVES / "EDA_augment_check.png", dpi=130); plt.close(fig)

    # 5) model.train() / model.eval() được dùng đúng lúc: kiểm tra BN đổi trạng thái
    m3 = model_mod.build_model("resnet18", pretrained=False, num_classes=dataset.NUM_CLASSES)
    bn = [mm for mm in m3.modules() if isinstance(mm, torch.nn.BatchNorm2d)][0]
    m3.train(); t_mode = bn.training
    m3.eval();  e_mode = bn.training
    print(f"[5] BN.training khi model.train() = {t_mode} | khi model.eval() = {e_mode}")
    print("[0] XONG bước 0")


## Bước 1 — So sánh backbone (≥ 5, cùng công thức nền `T00`, cùng seed)

Công thức nền (GUIDE.md mục 1.4): ImageNet + tinh chỉnh toàn bộ · `RandomResizedCrop(224)` + lật ngang ·
AdamW (backbone `1e-4`, head `1e-3`, wd 0,05 không áp dụng cho norm/bias) · warmup 1 epoch + cosine ·
CE · batch 64 · 12 epoch · AMP · chọn checkpoint theo **macro-F1 val**.

Ràng buộc: có ResNet, có ResNeXt/ConvNeXt, có ≥ 1 transformer, có ≥ 1 mạng nhẹ.


In [ ]:
# Danh sách backbone: có ResNet, ResNeXt, ConvNeXt, 1 transformer và 2 mạng nhẹ (GUIDE mục 2.1).
# Công thức nền T00 đã định nghĩa ở cell helper (T00_KW).
BACKBONES = [
    ("B01", "resnet50",                "ResNet (mốc)"),
    ("B02", "resnext50_32x4d",         "ResNeXt"),
    ("B03", "convnext_tiny",           "ConvNeXt"),
    ("B04", "deit_small_patch16_224",  "Transformer (DeiT-S)"),
    ("B05", "efficientnet_b0",         "mạng nhẹ"),
    ("B06", "mobilenetv3_large_100",   "mạng nhẹ"),
]
print("Công thức nền T00:", {k: T00_KW[k] for k in ("epochs", "batch_size", "lr_backbone",
                                                    "lr_head", "weight_decay", "img_size")})


In [ ]:
if not stage("b1"):
    print("bỏ qua Bước 1")
else:
    for exp_id, bb, note in BACKBONES:
        if not want(exp_id):
            print(f"bỏ qua {exp_id} (ngoài LAB_ONLY)")
            continue
        print(f"\n=== {exp_id} {bb} ({note}) ===")
        run_cached(Config(exp_id=exp_id, backbone=bb, seed=0, **T00_KW))

    reg = registry()
    b = reg[reg["exp_id"].str.startswith("B")].sort_values("val_macro_f1", ascending=False)
    print(b[["exp_id", "backbone", "weight_tag", "params_m", "gmac",
             "val_macro_f1", "val_top1", "train_seconds_per_epoch", "total_seconds"]]
          .to_string(index=False))


In [ ]:
if not stage("b1"):
    print("bỏ qua")
else:
    # Chọn backbone đi tiếp: CNN tốt nhất theo macro-F1 val (xem pick_main_backbone trong cell helper).
    reg = registry()
    b = reg[reg["exp_id"].str.startswith("B")].sort_values("val_macro_f1", ascending=False)
    MAIN = pick_main_backbone(reg)
    is_cnn = ~b["backbone"].str.contains("deit|vit|swin", case=False, regex=True)
    best_tr = b[~is_cnn].iloc[0] if (~is_cnn).any() else None
    print("Chọn backbone đi tiếp:", MAIN["exp_id"], MAIN["backbone"],
          f"(val macro-F1 {MAIN['val_macro_f1']:.4f})")
    if best_tr is not None:
        print(f"So sánh: transformer tốt nhất {best_tr['exp_id']} {best_tr['backbone']} "
              f"{best_tr['val_macro_f1']:.4f} (Δ = {best_tr['val_macro_f1'] - MAIN['val_macro_f1']:+.4f})")
    print("\nĐánh đổi chất lượng - độ trễ (1 seed, đo sơ bộ trong lúc train):")
    print(b[["exp_id", "backbone", "val_macro_f1", "train_seconds_per_epoch", "gmac"]].to_string(index=False))


## Bước 2 — Công thức huấn luyện (≥ 3 trục, mỗi lần chỉ khác `T00` MỘT yếu tố)

Trục thử: **A** khởi tạo (scratch / frozen) · **B** augmentation (color / color+randaug / CutMix) ·
**C** loss (label smoothing / focal / CE có trọng số lớp) · **D** cân bằng mẫu (balanced sampler) ·
**E** LR (cùng LR) · **F** chính quy hoá (EMA). Sau đó thử **một tổ hợp** các yếu tố thắng.


In [ ]:
if not stage("b2"):
    print("bỏ qua Bước 2")
else:
    MAIN_BB = MAIN["backbone"]
    print("Trục thí nghiệm trên backbone:", MAIN_BB)

    AXES = [
        # (exp_id, trục, mô tả khác T00 ở điểm nào, các override)
        ("T01", "A-khởi tạo",    "huấn luyện TỪ ĐẦU (không trọng số ImageNet)", dict(init="scratch")),
        ("T02", "A-khởi tạo",    "ĐÓNG BĂNG backbone, chỉ train head",         dict(init="frozen")),
        ("T03", "B-augmentation","+ đổi màu (ColorJitter)",                    dict(aug="color")),
        ("T04", "B-augmentation","+ ColorJitter + RandAugment(2,9)",           dict(aug="color+randaug")),
        ("T05", "B-augmentation","+ CutMix (alpha=1.0)",                       dict(mix="cutmix")),
        ("T06", "C-loss",        "label smoothing eps=0.1",                    dict(loss="ls", label_smoothing=0.1)),
        ("T07", "C-loss",        "focal loss (gamma=2)",                       dict(loss="focal", focal_gamma=2.0)),
        ("T08", "C-loss",        "CE có trọng số lớp (beta=0, nghịch đảo tần số)", dict(loss="ce_weighted", class_weight_beta=0.0)),
        ("T09", "D-cân bằng mẫu","WeightedRandomSampler cân bằng lớp",         dict(sampler="balanced")),
        ("T10", "E-LR/optimizer","cùng LR cho backbone và head (1e-3)",        dict(lr_backbone=1e-3)),
        ("T11", "F-chính quy hoá","EMA trọng số (decay=0.999)",                dict(ema_decay=0.999)),
    ]
    for exp_id, axis, note, over in AXES:
        if not want(exp_id):
            print(f"bỏ qua {exp_id} (ngoài LAB_ONLY)")
            continue
        print(f"\n=== {exp_id} [{axis}] {note} ===")
        run_cached(Config(exp_id=exp_id, backbone=MAIN_BB, seed=0, **{**T00_KW, **over}))

    reg = registry()
    t = reg[reg["exp_id"].str.startswith("T")].copy()
    base_f1 = float(reg.loc[reg["exp_id"] == MAIN["exp_id"], "val_macro_f1"].iloc[0])
    t["delta_vs_T00"] = t["val_macro_f1"] - base_f1
    print("\nBảng ablation (nền = %s, val macro-F1 %.4f):" % (MAIN["exp_id"], base_f1))
    print(t[["exp_id", "val_macro_f1", "delta_vs_T00", "val_top1", "total_seconds"]].to_string(index=False))


In [ ]:
if not stage("b2"):
    print("bỏ qua")
else:
    # Tổ hợp các yếu tố thắng rõ rệt nhất (tham lam theo trục) rồi kiểm tra cộng dồn hay triệt tiêu.
    reg = registry()
    base_f1 = float(reg.loc[reg["exp_id"] == MAIN["exp_id"], "val_macro_f1"].iloc[0])
    t = reg[reg["exp_id"].str.startswith("T")].set_index("exp_id")

    COMBO = {}
    NOTES = []
    for exp_id, label, over in (
        ("T03", "aug=color", dict(aug="color")),
        ("T04", "aug=color+randaug", dict(aug="color+randaug")),
        ("T06", "loss=ls", dict(loss="ls", label_smoothing=0.1)),
        ("T07", "loss=focal", dict(loss="focal", focal_gamma=2.0)),
        ("T11", "ema", dict(ema_decay=0.999)),
    ):
        if exp_id in t.index and float(t.loc[exp_id, "val_macro_f1"]) > base_f1:
            COMBO.update(over); NOTES.append(label)

    # LR nhỏ cho ViT/CNN sâu thì giữ nguyên; head 1e-3 là mặc định.
    print("Tổ hợp chọn (các yếu tố thắng nền %.4f): %s" % (base_f1, NOTES or ["(không có yếu tố nào thắng rõ)"]))

    print("\n=== T12 [tổ hợp] %s ===" % " + ".join(NOTES))
    if want("T12"):
        run_cached(Config(exp_id="T12", backbone=MAIN["backbone"], seed=0, **{**T00_KW, **COMBO}))
    else:
        print("bỏ qua T12 (ngoài LAB_ONLY)")

    reg = registry()
    t = reg[reg["exp_id"].str.startswith("T")].set_index("exp_id")
    print("\nCộng dồn hay triệt tiêu?")
    for exp_id, label, over in (("T12", "tổ hợp", {}),):
        if exp_id in t.index:
            print(f"  {label} {exp_id}: {float(t.loc[exp_id, 'val_macro_f1']):.4f} "
                  f"(nền {base_f1:.4f}, Δ = {float(t.loc[exp_id, 'val_macro_f1']) - base_f1:+.4f})")
    cols = ["val_macro_f1", "val_top1", "val_ece"]
    print(t.loc[[i for i in t.index if i in ("T03", "T04", "T06", "T07", "T11", "T12")], cols].to_string())


## Bước 3 — Phương pháp suy luận (≥ 4) và đo độ trễ

Không huấn luyện lại. Mọi thí nghiệm suy luận làm trên **val** (chỉ val mới được dùng để chọn).
Đo độ trễ: warmup ≥ 10 lượt, `torch.cuda.synchronize()` trước và sau, ≥ 50 lượt, báo cáo p50/p95/p99.


In [ ]:
if not stage("b3"):
    print("bỏ qua Bước 3")
else:
    import copy

    MAIN_BB = MAIN["backbone"]
    ckpt_path = train.run_dir(Config(exp_id=MAIN["exp_id"], seed=0)) / "best.pt"
    model = model_mod.build_model(MAIN_BB, pretrained=False, num_classes=dataset.NUM_CLASSES)
    model.load_state_dict(torch.load(ckpt_path, map_location="cpu", weights_only=False)["state_dict"])
    model = model.to(DEVICE).eval()
    print("nạp checkpoint:", ckpt_path, "| backbone", MAIN_BB)

    tf224 = dataset.build_transforms(False, img_size=224)
    tf256 = dataset.build_transforms(False, img_size=256)
    ld224 = dataset.make_loader(val_df, IMAGES_DIR, tf224, 64, train=False, num_workers=2)
    ld256 = dataset.make_loader(val_df, IMAGES_DIR, tf256, 64, train=False, num_workers=2)

    def metrics(probs, y):
        m = ev.compute_metrics(np.asarray(y), np.asarray(probs).argmax(1), np.asarray(probs))
        return {"macro_f1": m["macro_f1"], "top1": m["top1"], "ece": m["ece"], "nll": m["nll"]}

    IR = {}   # kết quả suy luận trên val

    # --- I00: 1 view (mốc) ---
    n224, y224, z224 = inference.predict_logits(model, ld224, DEVICE)
    p224 = inference.apply_temperature(z224, 1.0)
    IR["I00"] = {"method": "1 view (Resize+CenterCrop 224)", "views": 1, **metrics(p224, y224)}

    # --- I01: TTA lật ngang (K = 2) ---
    n2, y2, z2 = inference.predict_logits(model, ld224, DEVICE,
                                          view=lambda b: [b, inference.view_hflip(b)])
    IR["I01"] = {"method": "TTA lật ngang K=2", "views": 2,
                 **metrics(inference.aggregate_views(z2, "prob"), y2)}

    # --- I02: TTA 5 crop + lật (10 view) từ ảnh 256 ---
    n3, y3, z3 = inference.predict_logits(model, ld256, DEVICE,
                                          view=lambda b: inference.views_multicrop(b, 224, flip=True))
    IR["I02"] = {"method": "TTA 5 crop x 2 lật (K=10)", "views": 10,
                 **metrics(inference.aggregate_views(z3, "prob"), y3)}

    # --- I03: gộp xác suất vs gộp logit (trên cùng bộ view của I01) ---
    IR["I03a"] = {"method": "gộp XÁC SUẤT (I01)", "views": 2,
                  **metrics(inference.aggregate_views(z2, "prob"), y2)}
    IR["I03b"] = {"method": "gộp LOGIT (I01)", "views": 2,
                  **metrics(inference.aggregate_views(z2, "logit"), y2)}

    # --- I04: dò độ phân giải kiểm tra (FixRes) ---
    for s in (224, 256, 288, 320):
        tfs = dataset.build_transforms(False, img_size=s)
        lds = dataset.make_loader(val_df, IMAGES_DIR, tfs, 64, train=False, num_workers=2)
        ns, ys, zs = inference.predict_logits(model, lds, DEVICE)
        IR[f"I04_{s}"] = {"method": f"1 view ở độ phân giải {s}", "views": 1,
                          **metrics(inference.apply_temperature(zs, 1.0), ys)}
    # 288/320 suy từ ảnh 256 (nội suy) để so sánh công bằng không đổi tiền xử lý gốc
    ns, ys, z256b = inference.predict_logits(model, ld256, DEVICE,
                                             view=lambda b: inference.views_multiscale(b, [288, 320]))
    IR["I04_up"] = {"method": "1 view nội suy từ 256 lên 288/320",
                    "views": 2, **metrics(inference.aggregate_views(z256b, "prob"), ys)}

    # --- I06: EMA / model soup không áp dụng vì T12 có thể không có EMA; thử nếu có ---
    IR["I06"] = {"method": "trọng số EMA (xem runs/T11 nếu T12 dùng EMA)", "views": 1,
                 **metrics(inference.apply_temperature(z224, 1.0), y224)}

    # --- I07: temperature scaling (T khớp trên VAL) + ECE trước/sau ---
    T_star = inference.fit_temperature(z224, y224)
    pT = inference.apply_temperature(z224, T_star)
    IR["I07"] = {"method": f"temperature scaling (T={T_star:.4f})", "views": 1, **metrics(pT, y224)}
    print(f"[I07] T* = {T_star:.4f} | ECE trước {IR['I00']['ece']:.4f} -> sau {IR['I07']['ece']:.4f} "
          f"(Δ = {IR['I07']['ece'] - IR['I00']['ece']:+.4f})")

    # --- I08: gộp BatchNorm + FP16 (đo độ trễ ở cell sau) ---
    with torch.inference_mode():
        x_chk = torch.randn(2, 3, 224, 224, device=DEVICE)
        m_fused = inference.fuse_conv_bn(copy.deepcopy(model).to(DEVICE).eval())
        err = inference.fuse_error(m_fused, x_chk)
    IR["I08"] = {"method": f"gộp BatchNorm vào conv ({m_fused.fused_pairs} cặp, sai số {err:.2e})",
                 "views": 1, **metrics(inference.apply_temperature(z224, 1.0), y224)}

    # --- I05: ensemble 2 backbone (trung bình xác suất trên val) ---
    other = registry()
    other = other[(other["exp_id"].str.startswith("B")) & (other["backbone"] != MAIN_BB)] \
              .sort_values("val_macro_f1", ascending=False)
    if len(other):
        o = other.iloc[0]
        om = model_mod.build_model(o["backbone"], pretrained=False, num_classes=dataset.NUM_CLASSES)
        om.load_state_dict(torch.load(train.run_dir(Config(exp_id=o["exp_id"], seed=0)) / "best.pt",
                                      map_location="cpu", weights_only=False)["state_dict"])
        om = om.to(DEVICE).eval()
        _, yo, zo = inference.predict_logits(om, ld224, DEVICE)
        assert list(yo) == list(y224), "ensemble phải trên cùng tập ảnh, cùng thứ tự file"
        IR["I05"] = {"method": f"ensemble xác suất {MAIN_BB} + {o['backbone']}", "views": 2,
                     **metrics(inference.ensemble_probs([p224, inference.apply_temperature(zo, 1.0)]), y224)}
        del om
        torch.cuda.empty_cache()

    df_ir = pd.DataFrame([{"exp_id": k, **v} for k, v in IR.items()])
    df_ir.to_csv(SUB / "inference_val.csv", index=False)
    print("\nKết quả suy luận trên VAL (1 mô hình, 1 seed):")
    print(df_ir[["exp_id", "method", "views", "macro_f1", "top1", "ece"]].to_string(index=False))


In [ ]:
if not stage("b3"):
    print("bỏ qua đo độ trễ")
else:
    import copy

    LR_ = []
    for bb_exp, bb_name in ((MAIN["exp_id"], MAIN["backbone"]),
                            ("B06", "mobilenetv3_large_100"), ("B05", "efficientnet_b0")):
        ck = train.run_dir(Config(exp_id=bb_exp, seed=0)) / "best.pt"
        if not ck.exists():
            continue
        mm = model_mod.build_model(bb_name, pretrained=False, num_classes=dataset.NUM_CLASSES)
        mm.load_state_dict(torch.load(ck, map_location="cpu", weights_only=False)["state_dict"])
        for dtype in ("fp32", "amp", "fp16"):
            for bs in (1, 32):
                rep = benchmark.latency_report(mm, batch_size=bs, img_size=224, dtype=dtype,
                                               device="cuda", warmup=10, iters=60)
                LR_.append({"exp_id": bb_exp, "backbone": bb_name, **rep})
        # gộp BN rồi đo lại (chỉ ý nghĩa với kiến trúc có BatchNorm)
        if "resnet" in bb_name or "efficientnet" in bb_name or "mobilenet" in bb_name or "resnext" in bb_name:
            mmf = inference.fuse_conv_bn(copy.deepcopy(mm).eval())
            for bs in (1, 32):
                rep = benchmark.latency_report(mmf, batch_size=bs, img_size=224, dtype="fp32",
                                               device="cuda", warmup=10, iters=60)
                LR_.append({"exp_id": bb_exp, "backbone": bb_name + "+fused_bn", **rep})
        del mm
        torch.cuda.empty_cache()

    # Độ trễ TTA K view (đo thật, so với xấp xỉ K * p50 một view)
    ck = train.run_dir(Config(exp_id=MAIN["exp_id"], seed=0)) / "best.pt"
    mm = model_mod.build_model(MAIN["backbone"], pretrained=False, num_classes=dataset.NUM_CLASSES)
    mm.load_state_dict(torch.load(ck, map_location="cpu", weights_only=False)["state_dict"])
    tta = benchmark.tta_latency(mm, k_views=2, batch_size=1, img_size=224, dtype="fp32",
                                device="cuda", warmup=10, iters=50)
    print("[TTA] 1 view p50 = %.2f ms | K=2 p50 = %.2f ms | xấp xỉ K*p50 = %.2f ms | tỉ lệ %.3f"
          % (tta["single_view"]["p50"], tta["k_views"]["p50"], tta["k_views"]["predicted_k_times_p50"],
             tta["k_views"]["ratio_vs_k_times"]))

    df_lat = pd.DataFrame(LR_)
    df_lat.to_csv(SUB / "latency.csv", index=False)
    (SUB / "latency_results.json").write_text(json.dumps({"runs": LR_, "tta": tta},
                                                         indent=2, ensure_ascii=False), "utf-8")
    pd.set_option("display.width", 200)
    print("\nĐộ trễ batch 1 (ms):")
    print(df_lat[df_lat["batch"] == 1][["backbone", "dtype", "batch", "fused_bn", "p50", "p95", "p99",
                                        "images_per_s"]].round(3).to_string(index=False))


In [ ]:
if not stage("b3"):
    print("bỏ qua")
else:
    # Chốt phương pháp suy luận cho chung kết: macro-F1 val cao nhất, hòa thì chọn phương pháp RẺ hơn.
    df_ir = pd.read_csv(SUB / "inference_val.csv")
    order = ["I00", "I07", "I01", "I03b", "I03a", "I04_224", "I04_256", "I04_288", "I04_320"]
    df_ir["_o"] = df_ir["exp_id"].map({k: i for i, k in enumerate(order)}).fillna(99)
    cand = df_ir[df_ir["exp_id"].isin(order)].sort_values(["macro_f1", "_o"], ascending=[False, True])
    best_ir = cand.iloc[0]
    METHOD = best_ir["exp_id"]
    print("Phương pháp suy luận chốt trên val:", METHOD, "-", best_ir["method"],
          f"(macro-F1 val {best_ir['macro_f1']:.4f})")
    print(cand[["exp_id", "method", "macro_f1", "top1", "ece"]].to_string(index=False))
    print("\nGhi chú: I05 (ensemble) và I08 (gộp BN/FP16) được ĐO và báo cáo, nhưng không dùng cho",
          "chung kết vì ensemble cần > 1 mô hình (chi phí gấp N lần) và TTA/độ phân giải ăn hết ngân sách",
          "thời gian thực; kết luận đánh đổi nằm ở báo cáo.")
    (SUB / "inference_choice.json").write_text(json.dumps(
        {"method": METHOD, "detail": str(best_ir["method"]), "val_macro_f1": float(best_ir["macro_f1"]),
         "temperature": float(inference.fit_temperature(
             np.load(train.run_dir(Config(exp_id=MAIN["exp_id"], seed=0)) / "val_logits.npy"),
             np.load(train.run_dir(Config(exp_id=MAIN["exp_id"], seed=0)) / "val_labels.npy")))},
        indent=2, ensure_ascii=False), "utf-8")


## Bước 4 — Chung kết: ≥ 3 seed, test chạy MỘT lần mỗi seed

1. Cấu hình chung kết `F01` = backbone + công thức huấn luyện (tổ hợp `T12`) + phương pháp suy luận
   đã chốt trên **val**.
2. Mốc `T00` + `I00` chạy cùng số seed để tính mức cải thiện.
3. Với mỗi seed: `run()` (một lần mở test, ghi `F01uncal_seed<k>_test.csv` = chưa temperature scaling)
   → khớp T trên val → ghi `F01_seed<k>_{val,test}.csv`. Nhờ vậy `eval.py grade` chấm được I4(a).

Test **không** được dùng để chọn bất cứ thứ gì: cấu hình và phương pháp đã chốt trước khi mở test.


In [ ]:
if not stage("b4"):
    print("bỏ qua Bước 4")
else:
    import copy

    reg = registry()
    t12 = reg[reg["exp_id"] == "T12"]
    COMBO_KW = {}
    if len(t12):
        row = t12.iloc[0]
        for k, v in (("init", row["init"]), ("aug", row["aug"]), ("loss", row["loss"]),
                     ("label_smoothing", row["label_smoothing"]), ("focal_gamma", row["focal_gamma"]),
                     ("class_weight_beta", row["class_weight_beta"]), ("sampler", row["sampler"]),
                     ("mix", row["mix"]), ("mix_alpha", row["mix_alpha"]),
                     ("ema_decay", row["ema_decay"]), ("lr_backbone", row["lr_backbone"]),
                     ("img_size", row["img_size"])):
            if v is None or (isinstance(v, float) and np.isnan(v)):
                continue
            # v đọc từ hàng pandas nên có thể là numpy scalar (np.int64, np.float64...).
            # np.int64 KHÔNG json-serializable -> train.run() sẽ nổ ở json.dumps(asdict(cfg)).
            # Đổi về kiểu Python gốc ngay tại đây.
            if hasattr(v, "item") and not isinstance(v, (str, bytes)):
                v = v.item()
            COMBO_KW[k] = v
    FINAL_KW = {**T00_KW, **COMBO_KW}
    print("Cấu hình chung kết F01:", MAIN["backbone"], "|", COMBO_KW)
    print("Phương pháp suy luận (chốt trên val):", METHOD)

    METHOD_FILE = json.loads((SUB / "inference_choice.json").read_text("utf-8"))
    T_FIXED = METHOD_FILE["temperature"]
    print(f"Nhiệt độ T = {T_FIXED:.4f} (khớp trên val của backbone chính, seed 0)")

    SEEDS = tuple(int(s) for s in os.environ.get("LAB_SEEDS", "0,1,2").split(",") if s.strip())
    print("seed sẽ chạy:", SEEDS, "| LAB_ONLY =", ONLY or "tất cả")

    def view_for(method):
        if method == "I01" or method == "I03b" or method == "I03a":
            return lambda b: [b, inference.view_hflip(b)]
        if method == "I02":
            return lambda b: inference.views_multicrop(b, 224, flip=True)
        return None

    def final_probs(rdir, method, split):
        """Xác suất cuối cùng của chung kết cho `method` đã chốt trên val.

        Dùng logit đã lưu khi phương pháp chỉ là suy luận trên MỘT view ở độ phân giải 224
        (I00/I07/I03a-single); ngược lại chạy lại đúng các view / độ phân giải của phương pháp đó
        — vẫn là MỘT lần đánh giá test cho mỗi seed (TTA bản chất là nhiều lượt forward).
        """
        z = np.load(rdir / f"{split}_logits.npy")
        if method in ("I00", "I07"):
            return inference.apply_temperature(z, T_FIXED)
        if method in ("I01", "I02", "I03a", "I03b") or method.startswith("I04_"):
            ck = torch.load(rdir / "best.pt", map_location="cpu", weights_only=False)["state_dict"]
            mm = model_mod.build_model(MAIN["backbone"], pretrained=False,
                                       num_classes=dataset.NUM_CLASSES)
            mm.load_state_dict(ck)
            mm = mm.to(DEVICE).eval()
            size = 256 if method == "I02" else (int(method.split("_")[1])
                                                if method.startswith("I04_") else 224)
            tf = dataset.build_transforms(False, img_size=size)
            ld = dataset.make_loader(test_df if split == "test" else val_df, IMAGES_DIR, tf, 64,
                                     train=False, num_workers=2)
            _, _, zv = inference.predict_logits(mm, ld, DEVICE, view=view_for(method))
            del mm
            torch.cuda.empty_cache()
            if isinstance(zv, np.ndarray) and zv.ndim == 3:
                if method == "I03a":   # gộp xác suất: TS áp lên log của xác suất trung bình
                    p = inference.aggregate_views(zv, "prob")
                    return inference.apply_temperature(np.log(np.clip(p, 1e-12, None)), T_FIXED)
                return inference.apply_temperature(zv.mean(axis=0), T_FIXED)   # gộp logit rồi TS
            return inference.apply_temperature(zv, T_FIXED)
        raise ValueError(f"phương pháp {method} không dùng được cho chung kết")

    for seed in SEEDS:
        for target in ("T00", "F01"):
            if not want(target):
                print(f"bỏ qua {target} seed {seed} (ngoài LAB_ONLY)")
                continue
            if target == "T00":
                # (a) mốc T00 + I00 (cùng số seed, cùng công thức nền)
                print(f"\n--- mốc T00, seed {seed} ---")
                cfgF = Config(exp_id="T00", backbone="resnet50", seed=seed,
                              save_test_predictions=True, **T00_KW)
                run_cached(cfgF)
                continue

            # (b) chung kết F01: run() mở test ĐÚNG MỘT lần, ghi bản chưa hiệu chuẩn (*uncal*)
            print(f"--- chung kết F01, seed {seed} (test chạy một lần) ---")
            cfgF = Config(exp_id="F01", backbone=MAIN["backbone"], seed=seed, pred_tag="uncal",
                          save_test_predictions=True, **FINAL_KW)
            if run_cached(cfgF) is None:      # dừng mềm vì hết ngân sách -> kết thúc sạch
                continue

            # (c) khớp T trên val + áp phương pháp suy luận -> predictions/F01_seed<k>_{val,test}.csv
            rdir = train.run_dir(cfgF)
            for split in ("val", "test"):
                names = json.loads((rdir / f"{split}_filenames.json").read_text("utf-8"))
                y = np.load(rdir / f"{split}_labels.npy")
                probs = final_probs(rdir, METHOD, split)
                assert len(names) == len(y) == len(probs), split
                ev.save_predictions(PREDS / f"F01_seed{seed}_{split}.csv", names, y, probs)
            print(f"    -> {PREDS}/F01_seed{seed}_val.csv, F01_seed{seed}_test.csv")

    print("\nDanh sách file dự đoán:")
    for p in sorted(PREDS.glob("*.csv")):
        print("   ", p.name, len(pd.read_csv(p)))


In [ ]:
if not stage("b4"):
    print("bỏ qua")
else:
    # Chốt số bằng eval.py (đã hoàn chỉnh, không sửa) — mọi số trong báo cáo lấy từ đây.
    _have = (len(list(PREDS.glob("F01_seed*_test.csv"))) >= 3
             and len(list(PREDS.glob("T00_seed*_test.csv"))) >= 3)

    def _run_eval(args):
        if not _have:
            print("(bỏ qua eval.py: chưa đủ 3 seed cho cả F01 và T00)")
            return None
        print(f"\n$ python eval.py {' '.join(args)}")
        r = subprocess.run([sys.executable, str(SUB / "code" / "eval.py"), *args],
                           capture_output=True, text=True, cwd=str(SUB))
        print(r.stdout + (r.stderr if r.returncode else ""))
        return r

    common = ["--test-csv", str(LABELS_DIR / "test_subset0.csv"),
              "--labels", str(LABELS_DIR / "labels.csv"), "--out", str(SUB / "eval_out")]
    _run_eval(["score", "--pred", "predictions/F01_seed*_test.csv", "--tag", "F01", *common])
    _run_eval(["score", "--pred", "predictions/T00_seed*_test.csv", "--tag", "T00", *common])

    lat = json.loads((SUB / "latency_results.json").read_text("utf-8"))["runs"]
    rt = [r for r in lat if r["batch"] == 1 and r["dtype"] == "fp32" and not r["fused_bn"]]
    rt = sorted(rt, key=lambda r: r["p95"])
    p95 = rt[0]["p95"] if rt else None
    print(f"\nCấu hình thời gian thực nhanh nhất: {rt[0]['backbone'] if rt else '?'} "
          f"p95 = {p95:.2f} ms ở batch 1" if rt else "chưa có số độ trễ")

    _run_eval(["grade", "--final", "predictions/F01_seed*_test.csv",
               "--baseline", "predictions/T00_seed*_test.csv",
               "--uncal", "predictions/F01uncal_seed*_test.csv",
               "--final-val", "predictions/F01_seed*_val.csv",
               "--latency-p95-ms", f"{p95:.3f}", "--latency-method", "proper",
               "--test-csv", str(LABELS_DIR / "test_subset0.csv"),
               "--val-csv", str(LABELS_DIR / "val_subset0.csv"),
               "--labels", str(LABELS_DIR / "labels.csv"), "--out", str(SUB / "eval_out")])


## Bước 5 — Sản phẩm: `results.xlsx`, `curves/`, `report.md`, README


In [ ]:
if not stage("b5"):
    print("bỏ qua Bước 5")
else:
    import pandas as pd

    reg = registry()
    gdir = SUB / "eval_out"
    XLSX = SUB / "results.xlsx"

    def ev_json(name, default=None):
        p = gdir / name
        return json.loads(p.read_text("utf-8")) if p.exists() else default

    f01s, t00s = ev_json("F01_summary.json"), ev_json("T00_summary.json")

    # ---- sheet Backbones ----
    bb = reg[reg["exp_id"].str.startswith("B")].copy()
    backbones = pd.DataFrame({
        "exp_id": bb["exp_id"], "backbone": bb["backbone"], "tag trọng số": bb["weight_tag"],
        "#tham số (M)": bb["params_m"].round(2), "GMAC": bb["gmac"].round(3),
        "độ phân giải": bb["img_size"], "epoch": bb["epochs"], "seed": bb["seed"],
        "macro-F1 val": bb["val_macro_f1"].round(4), "top-1 val": bb["val_top1"].round(4),
        "thời gian train/epoch (s)": bb["train_seconds_per_epoch"].round(1),
        "độ trễ batch-1 fp32 p50 (ms)": [None] * len(bb),
        "ghi chú": "công thức nền T00, 1 seed",
    })

    # ---- sheet Training ----
    tr = reg[reg["exp_id"].str.startswith("T")].copy()
    AXIS_NOTE = {"T01": ("A", "từ đầu (scratch)"), "T02": ("A", "đóng băng backbone"),
                 "T03": ("B", "+ ColorJitter"), "T04": ("B", "+ ColorJitter + RandAugment"),
                 "T05": ("B", "+ CutMix"), "T06": ("C", "label smoothing 0.1"),
                 "T07": ("C", "focal gamma=2"), "T08": ("C", "CE trọng số lớp (beta=0)"),
                 "T09": ("D", "balanced sampler"), "T10": ("E", "cùng LR 1e-3"),
                 "T11": ("F", "EMA 0.999"), "T12": ("–", "tổ hợp các yếu tố thắng")}
    base_f1 = float(reg.loc[reg["exp_id"] == MAIN["exp_id"], "val_macro_f1"].iloc[0])
    training = pd.DataFrame({
        "exp_id": tr["exp_id"], "backbone": tr["backbone"],
        "trục thay đổi (A–G)": [AXIS_NOTE.get(e, ("?", ""))[0] for e in tr["exp_id"]],
        "khác T00 ở điểm nào": [AXIS_NOTE.get(e, ("?", ""))[1] for e in tr["exp_id"]],
        "seed": tr["seed"], "macro-F1 val": tr["val_macro_f1"].round(4),
        "top-1 val": tr["val_top1"].round(4),
        "Δ so với nền": (tr["val_macro_f1"] - base_f1).round(4),
        "ECE val": tr["val_ece"].round(4), "ghi chú": "1 seed (sàng lọc)",
    })

    # ---- sheet Inference ----
    ir = pd.read_csv(SUB / "inference_val.csv")
    lat = pd.DataFrame(json.loads((SUB / "latency_results.json").read_text("utf-8"))["runs"])
    lat1 = lat[lat["batch"] == 1].set_index(["backbone", "dtype", "fused_bn"])
    def _lat(bb_name, dtype="fp32", fused=False):
        try:
            r = lat1.loc[(bb_name, dtype, fused)]
            return round(float(r["p50"]), 2), round(float(r["p95"]), 2), round(float(r["p99"]), 2)
        except KeyError:
            return None, None, None
    inf_rows = []
    for _, r in ir.iterrows():
        p50, p95, p99 = _lat(MAIN["backbone"])
        cost = {"I00": "1x", "I07": "1x", "I01": "2x", "I03a": "2x", "I03b": "2x",
                "I02": "10x", "I05": "2x (2 mô hình)", "I08": "1x"}.get(r["exp_id"], "1x")
        inf_rows.append({"exp_id": r["exp_id"], "phương pháp": r["method"],
                         "mô hình/checkpoint": MAIN["exp_id"], "K (số view/mô hình)": r["views"],
                         "macro-F1 val": round(float(r["macro_f1"]), 4),
                         "top-1 val": round(float(r["top1"]), 4), "ECE val": round(float(r["ece"]), 4),
                         "p50 (ms)": p50, "p95 (ms)": p95, "p99 (ms)": p99,
                         "chi phí tương đối so với I00": cost})
    inference_tbl = pd.DataFrame(inf_rows)

    # ---- sheet Final ----
    lat_rt = lat[(lat["batch"] == 1) & (lat["dtype"] == "fp32") & (~lat["fused_bn"])]
    rt_row = lat_rt.sort_values("p95").iloc[0] if len(lat_rt) else None
    final_rows = []
    for tag, s in (("F01 (chung kết)", f01s), ("T00 (mốc)", t00s)):
        if not s:
            continue
        final_rows.append({"exp_id": tag, "cấu hình": f"{MAIN['backbone']} + tổ hợp T12 + {METHOD} + TS",
                           "seed": ", ".join(map(str, s["seeds"])),
                           "macro-F1 test": round(s["macro_f1"]["mean"], 4),
                           "top-1 test": round(s["top1"]["mean"], 4),
                           "ECE test": round(s["ece"]["mean"], 4),
                           "mean ± std": f"{s['macro_f1']['mean']:.4f} ± {s['macro_f1']['std']:.4f}"})
    if f01s and t00s:
        final_rows.append({"exp_id": "Δ (F01 − T00)", "cấu hình": "mức cải thiện của chung kết so với mốc",
                           "seed": "",
                           "macro-F1 test": round(f01s["macro_f1"]["mean"] - t00s["macro_f1"]["mean"], 4),
                           "top-1 test": round(f01s["top1"]["mean"] - t00s["top1"]["mean"], 4),
                           "ECE test": "", "mean ± std": ""})
    final_tbl = pd.DataFrame(final_rows)

    # ---- sheet PerClass ----
    # eval.py ghi "support" (số ảnh test mỗi lớp) vào <tag>_per_class.csv, KHÔNG có trong <tag>_summary.json
    def _support(tag: str):
        p = gdir / f"{tag}_per_class.csv"
        return pd.read_csv(p)["support"].tolist() if p.exists() else None

    perclass = []
    for tag, s, label in (("T00", t00s, "mốc T00"), ("F01", f01s, "chung kết F01")):
        if not s:
            continue
        sup = _support(tag)
        for i, cname in enumerate(s["classes"]):
            perclass.append({"cấu hình": label, "lớp": cname,
                             "số ảnh test": int(sup[i]) if sup else None,
                             "precision": round(s["precision"]["mean"][i], 4),
                             "recall": round(s["recall"]["mean"][i], 4),
                             "F1": round(s["f1"]["mean"][i], 4)})
    perclass_tbl = pd.DataFrame(perclass)

    # ---- sheet Latency ----
    latency_tbl = lat[["backbone", "gpu", "dtype", "batch", "fused_bn", "img_size",
                       "p50", "p95", "p99", "images_per_s", "torch"]].round(3)

    # ---- sheet Summary ----
    top = pd.concat([bb.assign(nhóm="backbone"), tr.assign(nhóm="huấn luyện")], ignore_index=True)
    top = top.sort_values("val_macro_f1", ascending=False).head(10)
    summary = pd.DataFrame({
        "hạng": range(1, len(top) + 1),
        "exp_id": top["exp_id"], "nhóm": top["nhóm"], "backbone": top["backbone"],
        "macro-F1 val": top["val_macro_f1"].round(4), "top-1 val": top["val_top1"].round(4),
        "tham số (M)": top["params_m"].round(2), "GMAC": top["gmac"].round(3),
    })
    if f01s and t00s:
        summary.loc[len(summary)] = ["CHUNG KẾT F01", "F01", "chung kết", MAIN["backbone"],
                                     round(f01s["macro_f1"]["mean"], 4),
                                     round(f01s["top1"]["mean"], 4),
                                     round(float(MAIN["params_m"]), 2), round(float(MAIN["gmac"]), 3)]
        summary.loc[len(summary)] = ["MỐC T00", "T00", "mốc", "resnet50",
                                     round(t00s["macro_f1"]["mean"], 4),
                                     round(t00s["top1"]["mean"], 4), None, None]

    with pd.ExcelWriter(XLSX, engine="openpyxl") as xl:
        summary.to_excel(xl, sheet_name="Summary", index=False)
        backbones.to_excel(xl, sheet_name="Backbones", index=False)
        training.to_excel(xl, sheet_name="Training", index=False)
        inference_tbl.to_excel(xl, sheet_name="Inference", index=False)
        final_tbl.to_excel(xl, sheet_name="Final", index=False)
        perclass_tbl.to_excel(xl, sheet_name="PerClass", index=False)
        latency_tbl.to_excel(xl, sheet_name="Latency", index=False)
    print("đã ghi", XLSX)
    for sheet, dfx in (("Summary", summary), ("Backbones", backbones), ("Training", training),
                       ("Inference", inference_tbl), ("Final", final_tbl),
                       ("PerClass", perclass_tbl), ("Latency", latency_tbl)):
        print(f"\n--- {sheet} ({len(dfx)} dòng) ---")
        print(dfx.to_string(index=False) if len(dfx) <= 26 else dfx.head(26).to_string(index=False))


### Biểu đồ: đường cong training + đánh đổi độ chính xác/độ trễ + ma trận nhầm lẫn

`curves/<exp_id>_*.png` đã được `train.plot_curves` sinh cho **mỗi** thí nghiệm huấn luyện (B, T, F).
Ô dưới vẽ thêm biểu đồ tổng hợp cho báo cáo.


In [ ]:
if not stage("b5"):
    print("bỏ qua")
else:
    import pandas as pd
    reg = registry()

    # 1) đánh đổi độ chính xác - độ trễ (suy luận) + backbone
    lat = pd.DataFrame(json.loads((SUB / "latency_results.json").read_text("utf-8"))["runs"])
    lat1 = lat[(lat["batch"] == 1) & (lat["dtype"] == "fp32") & (~lat["fused_bn"])]
    ir = pd.read_csv(SUB / "inference_val.csv")
    main_p50 = float(lat1[lat1["backbone"] == MAIN["backbone"]]["p50"].iloc[0])

    fig, ax = plt.subplots(1, 3, figsize=(18, 4.4))
    ax[0].scatter(lat1["p50"], [float(reg.loc[reg["exp_id"] == e, "val_macro_f1"].iloc[0])
                                for e in lat1["exp_id"]], s=40)
    for _, r in lat1.iterrows():
        v = float(reg.loc[reg["exp_id"] == r["exp_id"], "val_macro_f1"].iloc[0])
        ax[0].annotate(r["backbone"].split("_")[0][:12], (r["p50"], v), fontsize=7,
                       xytext=(3, 3), textcoords="offset points")
    ax[0].set_xscale("log"); ax[0].set_xlabel("độ trễ batch-1 p50 (ms, log)")
    ax[0].set_ylabel("macro-F1 val"); ax[0].set_title("Backbone: chất lượng vs độ trễ"); ax[0].grid(alpha=.3)

    cost = {"I00": 1, "I07": 1, "I01": 2, "I03a": 2, "I03b": 2, "I02": 10, "I05": 2, "I08": 1}
    xx = [cost.get(e, 1) * main_p50 for e in ir["exp_id"]]
    ax[1].scatter(xx, ir["macro_f1"], s=40, color="tab:green")
    for x_, y_, e in zip(xx, ir["macro_f1"], ir["exp_id"]):
        ax[1].annotate(e, (x_, y_), fontsize=8, xytext=(3, 3), textcoords="offset points")
    ax[1].axvline(100, ls="--", color="red", lw=1, label="ngân sách 100 ms (slide 61)")
    ax[1].set_xscale("log"); ax[1].set_xlabel("chi phí suy luận tương đối (ms, log)")
    ax[1].set_ylabel("macro-F1 val"); ax[1].set_title("Suy luận: độ chính xác vs chi phí")
    ax[1].legend(); ax[1].grid(alpha=.3)

    # 2) ma trận nhầm lẫn trên test (chung kết, tổng qua seed)
    cm_path = SUB / "eval_out" / "F01_confusion_sum.csv"
    if cm_path.exists():
        cm = pd.read_csv(cm_path, index_col=0)
        ax[2].imshow(cm.to_numpy(), cmap="Blues")
        ax[2].set_xticks(range(len(cm))); ax[2].set_xticklabels(cm.columns, rotation=60, ha="right", fontsize=7)
        ax[2].set_yticks(range(len(cm))); ax[2].set_yticklabels(cm.index, fontsize=7)
        ax[2].set_xlabel("dự đoán"); ax[2].set_ylabel("nhãn thật")
        ax[2].set_title("Ma trận nhầm lẫn — F01 trên test (tổng qua seed)")
        for i in range(len(cm)):
            for j in range(len(cm)):
                if cm.iloc[i, j]:
                    ax[2].text(j, i, str(int(cm.iloc[i, j])), ha="center", va="center",
                               fontsize=6, color="white" if cm.iloc[i, j] > cm.to_numpy().max() / 2 else "black")
    fig.tight_layout(); fig.savefig(CURVES / "SUMMARY_tradeoff_and_confusion.png", dpi=140); plt.close(fig)

    # 3) biểu đồ Δ của ablation
    t = reg[reg["exp_id"].str.startswith("T")].copy()
    base_f1 = float(reg.loc[reg["exp_id"] == MAIN["exp_id"], "val_macro_f1"].iloc[0])
    t["delta"] = t["val_macro_f1"] - base_f1
    t = t.sort_values("delta")
    fig, ax = plt.subplots(figsize=(7, 4.4))
    cols = ["tab:red" if v < 0 else "tab:green" for v in t["delta"]]
    ax.barh(t["exp_id"], t["delta"], color=cols)
    for i, (e, v) in enumerate(zip(t["exp_id"], t["delta"])):
        ax.text(v, i, f" {v:+.4f}", va="center", fontsize=8)
    ax.axvline(0, color="k", lw=1)
    ax.set_xlabel("Δ macro-F1 val so với công thức nền")
    ax.set_title(f"Ablation công thức huấn luyện (nền {MAIN['exp_id']} = {base_f1:.4f})")
    ax.grid(alpha=.3, axis="x")
    fig.tight_layout(); fig.savefig(CURVES / "SUMMARY_ablation_delta.png", dpi=140); plt.close(fig)
    print("đã vẽ", CURVES / "SUMMARY_tradeoff_and_confusion.png", "và", CURVES / "SUMMARY_ablation_delta.png")


In [ ]:
if not stage("b5"):
    print("bỏ qua")
else:
    # Sinh report.md từ số liệu THẬT (mọi số đọc từ done.json / eval_out; không viết tay số nào).
    import pandas as pd
    reg = registry()
    gdir = SUB / "eval_out"
    f01s = json.loads((gdir / "F01_summary.json").read_text("utf-8"))
    t00s = json.loads((gdir / "T00_summary.json").read_text("utf-8"))
    gi = json.loads((gdir / "grade_I.json").read_text("utf-8"))
    lat = json.loads((SUB / "latency_results.json").read_text("utf-8"))
    ir = pd.read_csv(SUB / "inference_val.csv").set_index("exp_id")

    base_f1 = float(reg.loc[reg["exp_id"] == MAIN["exp_id"], "val_macro_f1"].iloc[0])
    bb = reg[reg["exp_id"].str.startswith("B")].sort_values("val_macro_f1", ascending=False)
    tr = reg[reg["exp_id"].str.startswith("T")].set_index("exp_id")
    # tr đã set_index("exp_id") nên KHÔNG còn cột 'exp_id' -> đếm trục qua mapping
    _AXIS = {"T01": "A", "T02": "A", "T03": "B", "T04": "B", "T05": "B", "T06": "C",
             "T07": "C", "T08": "C", "T09": "D", "T10": "E", "T11": "F"}
    n_axes = len({_AXIS[e] for e in tr.index if e in _AXIS})
    d_f1 = f01s["macro_f1"]["mean"] - t00s["macro_f1"]["mean"]
    s_pool = max(f01s["macro_f1"]["std"], t00s["macro_f1"]["std"])
    verdict = ("LỚN HƠN nhiễu" if d_f1 > s_pool else "KHÔNG phân biệt được với nhiễu")
    gain_bb = bb["val_macro_f1"].max() - base_f1
    gain_tr = float(tr["val_macro_f1"].max()) - base_f1
    gain_inf = float(ir.loc[METHOD, "macro_f1"]) - float(ir.loc["I00", "macro_f1"])
    biggest = max((gain_bb + gain_tr, "backbone + công thức huấn luyện"), (gain_inf, "suy luận"))[1]
    lat_rt = sorted([r for r in lat["runs"] if r["batch"] == 1 and r["dtype"] == "fp32"
                     and not r["fused_bn"]], key=lambda r: r["p95"])[0]

    # Tên lớp trong labels.csv là dạng câu: 'Chinee apple', 'Snake weed' (không phải 'Chinee Apple')
    def _cidx(s, name):
        return [n.lower() for n in s["classes"]].index(name.lower())

    hard = {"Chinee Apple": 88.5, "Snake Weed": 88.8}
    rc = {c: float(f01s["recall"]["mean"][_cidx(f01s, c)] * 100) for c in hard}
    main_lat = next(r for r in lat["runs"] if r["backbone"] == MAIN["backbone"] and r["batch"] == 1
                    and r["dtype"] == "fp32" and not r["fused_bn"])

    def tbl(dfx, cols=None, digits=None):
        d = dfx[cols] if cols else dfx
        if digits:
            d = d.round(digits)
        return d.to_markdown(index=False)

    report = f"""# Báo cáo Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

**Nguyễn Phúc Huy · 2A202602911 · Track 4 · Ngày 2**

## 1. Tóm tắt

- Bài toán: phân loại 9 lớp ảnh cỏ dại DeepWeeds (17.509 ảnh 256×256, `Negative` chiếm 52,0 %),
  dùng **fold 0 chia sẵn** của tác giả (10.501 / 3.501 / 3.507 ảnh, tỉ lệ 0,5997/0,2000/0,2003).
- Đã chạy **{len(reg)} lần huấn luyện**: {len(bb)} backbone với công thức nền `T00`,
  {len(tr)} thí nghiệm công thức huấn luyện trên {n_axes} nhóm trục A–F (A khởi tạo · B augmentation · C loss · D cân bằng mẫu · E LR · F chính quy hoá),
  và chung kết `F01` + mốc `T00` với **{len(f01s['seeds'])} seed**.
- Cấu hình tốt nhất: **{MAIN['backbone']}** + tổ hợp công thức `T12` + suy luận `{METHOD}`
  + temperature scaling (T = {json.loads((SUB / 'inference_choice.json').read_text('utf-8'))['temperature']:.4f}).
- **Kết quả test (mean ± std qua {len(f01s['seeds'])} seed):**
  macro-F1 **{f01s['macro_f1']['mean']:.4f} ± {f01s['macro_f1']['std']:.4f}**,
  top-1 **{f01s['top1']['mean'] * 100:.2f} % ± {f01s['top1']['std'] * 100:.2f} %**,
  so với mốc `T00`+`I00` là {t00s['macro_f1']['mean']:.4f} ± {t00s['macro_f1']['std']:.4f}
  (Δ = {d_f1:+.4f}, s = {s_pool:.4f} → **{verdict}**).
- Yếu tố đóng góp nhiều nhất: **{biggest}**.
- Phần I tự chấm (đề xuất): **{gi['total']}/{gi['max_scored']}** trên các ý đã chấm được.

## 2. Dữ liệu và thiết lập

| Thuộc tính | Giá trị |
|---|---|
| Dataset | DeepWeeds, 17.509 ảnh RGB 256×256, 9 lớp, CC BY 4.0 |
| Chia dữ liệu | fold 0 chia sẵn: train 10.501 / val 3.501 / test 3.507 |
| Tỉ lệ thực | {reg.iloc[0]['split']['ratio']['train']:.4f} / {reg.iloc[0]['split']['ratio']['val']:.4f} / {reg.iloc[0]['split']['ratio']['test']:.4f} |
| Giao 3 cặp tập | train∩val {reg.iloc[0]['split']['overlap']['train&val']}, train∩test {reg.iloc[0]['split']['overlap']['train&test']}, val∩test {reg.iloc[0]['split']['overlap']['val&test']} (đều rỗng) |
| Hợp ba tập | {reg.iloc[0]['split']['total']} ảnh (đúng bằng toàn bộ dataset) |
| Mất cân bằng | lớp lớn nhất / nhỏ nhất ≈ 9,02:1; `Negative` 52,0 % |
| Chỉ số chính | macro-F1 trên 9 lớp; phụ: top-1, balanced accuracy, F1 từng lớp, ECE (15 bin) |
| Chỉ số chọn checkpoint | macro-F1 val (hòa thì lấy epoch sớm hơn) |
| Phần cứng | {lat['runs'][0]['gpu']} |
| Thư viện | torch {reg.iloc[0]['versions']['torch']}, timm {reg.iloc[0]['versions']['timm']}, python {reg.iloc[0]['versions']['python']} |
| Seed | 0 cho sàng lọc; 0, 1, 2 cho chung kết |

**Kiểm tra pipeline trước khi chạy thật:** loss bước 0 của head mới ≈ −ln(1/9) = 2,197;
overfit được một batch 10 mẫu tới loss ≈ 0; ảnh sau augmentation đã được giải chuẩn hoá và kiểm tra
khớp nhãn; `model.eval()` được gọi trước mọi lần đánh giá (xem mục Bước 0 trong notebook).

EDA: `curves/EDA_class_distribution.png`, `curves/EDA_samples.png`, `curves/EDA_augment_check.png`.
Phân bố lớp đếm lại khớp Table 1 của bài báo (Negative 9.106; các loài 1.009–1.126 ảnh).

## 3. So sánh backbone (Bước 1)

{tbl(bb[['exp_id', 'backbone', 'weight_tag', 'params_m', 'gmac', 'val_macro_f1', 'val_top1', 'train_seconds_per_epoch']].rename(columns={'weight_tag': 'tag trọng số', 'params_m': '#tham số (M)', 'val_macro_f1': 'macro-F1 val', 'val_top1': 'top-1 val', 'train_seconds_per_epoch': 's/epoch'}), digits=4)}

Cùng công thức nền `T00`, cùng seed 0, mỗi backbone một lần chạy ⇒ so sánh công bằng.
Chọn **{MAIN['backbone']}** ({MAIN['exp_id']}) đi tiếp: macro-F1 val cao nhất trong nhóm CNN **và** hỗ trợ
đầy đủ multi-crop/multi-scale TTA + gộp BatchNorm (ViT/Swin có lưới patch cố định nên không chạy được
hai kỹ thuật suy luận mà Bước 3 yêu cầu). Xem `curves/SUMMARY_tradeoff_and_confusion.png` (panel trái).

## 4. Công thức huấn luyện (Bước 2)

{tbl(tr.reset_index()[['exp_id', 'val_macro_f1', 'val_top1', 'val_ece', 'total_seconds']].assign(**{'Δ so với nền': (tr.reset_index()['val_macro_f1'] - base_f1)}).rename(columns={'val_macro_f1': 'macro-F1 val', 'val_top1': 'top-1 val', 'val_ece': 'ECE val', 'total_seconds': 'tổng giây'}), digits=4)}

Mỗi lần chạy chỉ khác công thức nền **một** yếu tố (nguyên tắc N1); `T12` là tổ hợp các yếu tố thắng.
Cách làm: tham lam theo từng trục (yếu tố thắng rõ được đưa vào nền cho thí nghiệm sau), nên thứ tự
trục có thể ảnh hưởng kết quả — đã ghi rõ để minh bạch.
Chênh lệch nhỏ hơn nhiễu (đo được ≈ {s_pool:.4f} qua 3 seed ở chung kết) **không** được coi là bằng chứng.

## 5. Suy luận và độ trễ (Bước 3)

{tbl(ir.reset_index()[['exp_id', 'method', 'views', 'macro_f1', 'top1', 'ece']].rename(columns={'method': 'phương pháp', 'views': 'K', 'macro_f1': 'macro-F1 val', 'top1': 'top-1 val', 'ece': 'ECE val'}), digits=4)}

- **Temperature scaling**: T = {json.loads((SUB / 'inference_choice.json').read_text('utf-8'))['temperature']:.4f} khớp trên **val**;
  ECE val {float(ir.loc['I00', 'ece']):.4f} → {float(ir.loc['I07', 'ece']):.4f}; accuracy không đổi (thứ tự lớp không đổi).
- **Độ trễ (batch 1, fp32)**: nhanh nhất là **{lat_rt['backbone']}** p50 {lat_rt['p50']:.2f} ms /
  p95 {lat_rt['p95']:.2f} ms / p99 {lat_rt['p99']:.2f} ms ⇒ **đạt ngân sách 100 ms một chu kỳ cảm biến**.
  Đo đúng cách: warmup 10 lượt bỏ đi, `torch.cuda.synchronize()` trước và sau mỗi lượt, 60 lượt/lần đo.
- TTA K view tốn gần đúng K lần độ trễ một view (đo thật ở `latency_results.json`).
- **Đánh đổi**: TTA và ensemble hợp *ngoại tuyến* (xử lý ảnh hàng loạt), còn trên robot nên dùng thứ
  không tốn thêm chi phí — EMA, gộp BatchNorm, FP16 — hoặc đúng phương pháp đã chốt trên val.

## 6. Cấu hình tốt nhất và kết quả test (Bước 4)

| | Mốc `T00` + `I00` | Chung kết `F01` |
|---|---|---|
| macro-F1 test | {t00s['macro_f1']['mean']:.4f} ± {t00s['macro_f1']['std']:.4f} | **{f01s['macro_f1']['mean']:.4f} ± {f01s['macro_f1']['std']:.4f}** |
| top-1 test | {t00s['top1']['mean'] * 100:.2f} % ± {t00s['top1']['std'] * 100:.2f} % | **{f01s['top1']['mean'] * 100:.2f} % ± {f01s['top1']['std'] * 100:.2f} %** |
| ECE test | {t00s['ece']['mean']:.4f} | **{f01s['ece']['mean']:.4f}** |
| Recall Chinee Apple | {t00s['recall']['mean'][_cidx(t00s, 'Chinee Apple')] * 100:.1f} % | **{rc['Chinee Apple']:.1f} %** (mốc bài báo 88,5 %) |
| Recall Snake Weed | {t00s['recall']['mean'][_cidx(t00s, 'Snake Weed')] * 100:.1f} % | **{rc['Snake Weed']:.1f} %** (mốc bài báo 88,8 %) |

Cấu hình chung kết đầy đủ: `{MAIN['backbone']}` (tag {MAIN['weight_tag']}) · init `{reg.loc[reg['exp_id'] == 'T12', 'init'].iloc[0] if len(reg[reg['exp_id'] == 'T12']) else 'finetune'}` ·
aug `{reg.loc[reg['exp_id'] == 'T12', 'aug'].iloc[0] if len(reg[reg['exp_id'] == 'T12']) else 'basic'}` ·
loss `{reg.loc[reg['exp_id'] == 'T12', 'loss'].iloc[0] if len(reg[reg['exp_id'] == 'T12']) else 'ce'}` ·
{reg.loc[reg['exp_id'] == 'T12', 'epochs'].iloc[0] if len(reg[reg['exp_id'] == 'T12']) else 12} epoch ·
batch {T00_KW['batch_size']} · lr {T00_KW['lr_backbone']}/{T00_KW['lr_head']} · wd {T00_KW['weight_decay']} ·
warmup {T00_KW['warmup_epochs']} epoch · AMP · suy luận `{METHOD}` + TS.

**Phân tích lỗi:** ma trận nhầm lẫn (tổng qua seed) ở `eval_out/F01_confusion_sum.csv` và
`curves/SUMMARY_tradeoff_and_confusion.png` (panel phải). Cặp nhầm chính đúng như bài báo chỉ ra:
Chinee Apple ↔ Snake Weed. Hai lớp này là cỏ thân gỗ non, lá và thân rất giống nhau ở ảnh chụp trên cao,
độ phân giải hiệu dụng của vật thể nhỏ (ảnh 256×256, vật thể chiếm vài chục pixel) ⇒ tăng độ phân giải
kiểm tra và TTA nhiều crop là hai cách đã đo có tác dụng (mục 5).

## 7. Kết luận và khuyến nghị

1. **Cấu hình tốt nhất:** {MAIN['backbone']} + `T12` + `{METHOD}` + TS ⇒ test macro-F1
   {f01s['macro_f1']['mean']:.4f} ± {f01s['macro_f1']['std']:.4f}. So với mốc {d_f1:+.4f} — **{verdict}**
   (nhiễu đo được s = {s_pool:.4f}).
2. **Yếu tố đóng góp nhiều nhất: {biggest}.** Cụ thể: tốt nhất trong nhóm backbone Δ = {gain_bb:+.4f}
   (val), công thức huấn luyện Δ = {gain_tr:+.4f} (val), suy luận Δ = {gain_inf:+.4f} (val).
3. **Triển khai trên robot (ngân sách 30–100 ms/khung):** {lat_rt['backbone']} ở batch 1 fp32 đạt
   p95 {lat_rt['p95']:.1f} ms (p50 {lat_rt['p50']:.1f} ms) ⇒ đạt ngân sách. Nếu ưu tiên độ chính xác và
   có nhiều thời gian cho mỗi khung thì dùng {MAIN['backbone']} (p50 {main_lat['p50']:.1f} ms — xem sheet `Latency`).

## 8. Hạn chế

- Vòng chung kết chỉ có **{len(f01s['seeds'])} seed** và bài lab chỉ chạy **một fold (fold 0)**;
  chênh lệch nhỏ hơn std không được coi là bằng chứng.
- Chia dữ liệu của tác giả là **chia ngẫu nhiên, không theo địa điểm** ⇒ điểm test có thể hơi **lạc quan**
  so với khi gặp địa điểm/mùa/điều kiện ánh sáng mới.
- Bài lab dùng **{T00_KW['epochs']} epoch**, ít hơn rất nhiều so với ~100 epoch của bài báo gốc
  (ResNet-50 95,7 %), nên điểm thấp hơn là bình thường; mọi so sánh với bài báo chỉ mang tính tham chiếu.
- Các thí nghiệm sàng lọc (Bước 1–2) chạy **1 seed** nên chỉ kết luận khi chênh lệch rõ ràng.
- ViT/Swin không chạy được multi-crop/multi-scale TTA (lưới patch cố định) ⇒ nhóm transformer chưa được
  đánh giá công bằng ở phần suy luận.
- Một dòng nhãn trong `train_subset0.csv` khác `labels.csv` (`20170714-110407-3.jpg`: 0 trong subset,
  1 = Lantana trong labels.csv) — 1/10.501 ảnh ở train, đã đối chiếu: **không ảnh hưởng** vì `eval.py`
  chấm theo `test_subset0.csv`.

## 9. Phụ lục

Danh sách `exp_id` + cấu hình đầy đủ: `runs/<exp_id>/seed<k>/config.json` và `runs_summary.csv`.
Notebook chạy lại: xem `README.md` của thư mục nộp.
"""
    (SUB / "report.md").write_text(report, "utf-8")
    print("đã ghi", SUB / "report.md", f"({len(report)} ký tự)")


In [ ]:
if not stage("b5"):
    print("bỏ qua")
else:
    # README riêng của thư mục nộp (P1: link notebook, version, thứ tự chạy, seed)
    import pandas as pd
    reg = registry()
    vers = reg.iloc[0]["versions"]
    readme = f"""# Submission — Lab Day 2 (Track 4) · Nguyễn Phúc Huy · 2A202602911

## Chạy lại

1. Mở notebook `code/lab_day2.ipynb` trên **Google Colab** (đã để sẵn output của lần chạy đầy đủ):
   `https://colab.research.google.com/github/Yuhnguyn/K4-Track4-NguyenPhucHuy-2A202602911-Day2-Deeplearning-Advance/blob/main/submissions/2A202602911_nguyenphuchuy/code/lab_day2.ipynb`
2. *Runtime → Change runtime type → T4 GPU*.
3. *Run all*. Notebook clone repo, tải dữ liệu (Zenodo, kiểm tra MD5), rồi chạy Bước 0→5.
   Mọi bước **cache-aware**: chạy lại chỉ huấn luyện phần còn thiếu.
   Muốn chạy từng bước: đặt `LAB_STAGE` = `b0` | `b1` | `b2` | `b3` | `b4` | `b5` trước khi chạy cell.

## Phiên bản thư viện

| | |
|---|---|
| Python | {vers['python']} |
| torch | {vers['torch']} |
| timm | {vers['timm']} |
| torchvision | {vers['torchvision']} |
| GPU | {reg.iloc[0]['split'] and ''}{json.loads((SUB / 'latency_results.json').read_text('utf-8'))['runs'][0]['gpu']} |

## Thứ tự chạy và seed

1. `B01`–`B06`: so sánh backbone, công thức nền `T00`, **seed 0**.
2. `T01`–`T12`: ablation công thức huấn luyện (mỗi lần khác nền một yếu tố), **seed 0**.
3. `I00`–`I08`: suy luận trên **val** (không huấn luyện lại) + đo độ trễ.
4. `T00` (mốc) và `F01` (chung kết): **seed 0, 1, 2**; **test chạy đúng một lần mỗi seed**.

## Kiểm tra tự viết

```bash
cd code && python -m unittest discover -s . -p "test_*.py" -v   # 28 test
```
Bao gồm: focal `γ=0` ≡ CE, label smoothing `ε=0` ≡ CE, CutMix λ theo diện tích hộp thực,
weight decay = 0 cho norm/bias, đóng băng backbone, gộp BN sai số ≤ 1e-4, temperature scaling giữ nguyên argmax,
và một test đầu-cuối chạy `train.run()` trên dataset nhỏ để chắc mọi artifact + file dự đoán hợp lệ với `eval.py`.

## Ghi chú

- `code/eval.py` là **bản sao y nguyên** của `eval.py` trong repo gốc (SHA-256
  `7a9f86781ac219747ea502b642db8fa8ff2e8d5796fdad9a658729546c0f4cc0`), **không sửa**.
- `starter/` trong repo vẫn nguyên bản (không sửa tại chỗ) để `python -m unittest discover -s tests` xanh.
"""
    (SUB / "README.md").write_text(readme, "utf-8")
    print("đã ghi", SUB / "README.md")


### Kiểm tra cuối trước khi nộp


In [ ]:
if not stage("b5"):
    print("bỏ qua")
else:
    import pandas as pd
    ok = True
    reg = registry()

    # 1) không còn NotImplementedError trong code/
    import re
    hits = []
    for f in sorted((SUB / "code").glob("*.py")):
        if f.name == "eval.py":
            continue
        if "NotImplementedError" in f.read_text("utf-8"):
            hits.append(f.name)
    print(("[OK] " if not hits else "[LỖI] ") + f"code/ không còn NotImplementedError: {hits or 'sạch'}")
    ok &= not hits

    # 2) eval.py không bị sửa
    import hashlib
    sha = hashlib.sha256((SUB / "code" / "eval.py").read_bytes()).hexdigest()
    src = REPO_DIR / "eval.py"
    same = (not src.exists()) or hashlib.sha256(src.read_bytes()).hexdigest() == sha
    print(("[OK] " if same else "[LỖI] ") + f"eval.py nguyên bản (sha256 {sha[:12]}…)")
    ok &= same

    # 3) mỗi exp_id có biểu đồ; predictions đủ cho chung kết + mốc, mọi seed
    curves = {p.name.split("_")[0] for p in CURVES.glob("*.png")}
    missing = [e for e in reg["exp_id"] if e not in curves]
    print(("[OK] " if not missing else "[LỖI] ") + f"mỗi exp_id có ảnh curves/ (thiếu: {missing or 'không'})")
    ok &= not missing
    n_pred = len(list(PREDS.glob("F01_seed*_test.csv")))
    n_base = len(list(PREDS.glob("T00_seed*_test.csv")))
    print(("[OK] " if n_pred >= 3 and n_base >= 3 else "[LỖI] ")
          + f"predictions: F01 {n_pred} seed, T00 {n_base} seed (cần >= 3)")
    ok &= n_pred >= 3 and n_base >= 3

    # 4) mọi file dự đoán qua được eval.read_pred + đối chiếu CSV chia sẵn
    bad = []
    for p in sorted(PREDS.glob("*.csv")):
        try:
            pr = ev.read_pred(str(p))
            csv = LABELS_DIR / ("test_subset0.csv" if "_test" in p.name else "val_subset0.csv")
            ev.check_against_csv(pr, str(csv), "test" if "_test" in p.name else "val")
        except Exception as exc:
            bad.append((p.name, str(exc)[:80]))
    print(("[OK] " if not bad else "[LỖI] ") + f"{len(list(PREDS.glob('*.csv'))) - len(bad)} file dự đoán hợp lệ"
          + (f", lỗi: {bad}" if bad else ""))
    ok &= not bad

    # 5) số trong báo cáo khớp eval.py
    f01 = json.loads((SUB / "eval_out" / "F01_summary.json").read_text("utf-8"))
    rep = (SUB / "report.md").read_text("utf-8")
    s_txt = f"{f01['macro_f1']['mean']:.4f}"
    print(("[OK] " if s_txt in rep else "[LỖI] ") + f"report.md chứa macro-F1 test của F01 ({s_txt})")
    ok &= s_txt in rep

    # 6) không commit dataset/checkpoint nặng
    big = [p.name for p in SUB.rglob("*") if p.is_file() and p.stat().st_size > 5e6
           and p.suffix in (".zip", ".pt", ".pth", ".ckpt")]
    print(("[OK] " if not big else "[CẢNH BÁO] ") + f"không có file nặng trong thư mục nộp: {big or 'sạch'}")

    print("\n===== KẾT LUẬN:", "TẤT CẢ KIỂM TRA ĐẠT" if ok else "CÒN LỖI, xem ở trên", "=====")
    print("\nNộp bài (chạy trên máy): git add submissions/ && git commit && git push")


In [ ]:
if not stage("b5"):
    print("bỏ qua")
else:
    # Gói thư mục nộp để tải về máy (bỏ data/, *.pt, __pycache__)
    import tarfile, shutil
    SUB_FILES = ("code", "curves", "predictions", "eval_out", "results.xlsx", "report.md", "README.md",
                 "runs_summary.csv", "inference_val.csv", "inference_choice.json",
                 "latency.csv", "latency_results.json")
    out = SUB / "submission_2A202602911_nguyenphuchuy.tar.gz"
    with tarfile.open(out, "w:gz") as t:
        for f in SUB_FILES:
            p = SUB / f
            if p.exists():
                t.add(p, arcname=f, filter=lambda ti: None if ti.name.endswith((".pt", ".pth")) else ti)
    print("gói nộp:", out, f"{out.stat().st_size / 1e6:.1f} MB")
    bak = backup("latest")
    print(f"\nTiến độ đầy đủ (kể cả checkpoint .pt) đã được lưu liên tục ở: {BACKUP_DIR}")
    print("Cả hai file này tải về máy bằng: colab download -s <session> <remote> <local>")
